# How to use AMBRIC

In [ ]:
# | echo: false
import matplotlib_inline.backend_inline

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

Let's import the package.

In [ ]:
from ambric import Ambric
from ambric.utilities import generate_realistic_simulated_data

As a user, we have to bring a few things to the party. The first of course is data, which we will simulated data. (We'll specify how many underlying factors are driving regional dynamics here.)

In [ ]:
n_factors = 2
df = generate_realistic_simulated_data(n_factors=n_factors)

Data that are input into the model must have this structure:

In [ ]:
df.sample(10)

The user must also specify the details of these, saying which regional variables to use, which macroeconomic indicators to use (these should always have the aggregate region/top level geography as their region), and which extra regional covariates to use. We'll just use all of these:

In [ ]:
aggregation_region = "uk"
region_names = [x for x in df["region"].unique() if x != aggregation_region]
macro_names = [x for x in df["measure"].unique() if "macro" in x]
region_covariate_names = [x for x in df["measure"].unique() if "regional_covar" in x]

Gotcha: you must ensure that the annual regional data have datetime index entries for **all** quarters up to the last published annual value. For non-year end quarters, the values should be nan.

Okay, we're ready to build a **AMBRIC** model!

In [ ]:
amb = Ambric(
    df,
    macro_names,
    region_names,
    region_covariate_names,
    n_factors=n_factors,
)
amb

Note that the model has specified all of its details, including that it sees that there are 6 rows of the regional data missing that will be estimated by the model. The model also tell us it isn't fitted, so let's sort that. We'll choose a small number of iterations and posterior samples to keep the demo quick.

In [ ]:
n_iterations = 100000
n_posterior_samples = 1000
amb.fit(n_iterations, n_posterior_samples)

That's it! It's done. Now let's look at some results.

First, our regional estimates of quarterly growth must be consistent with the observed national growth. We can check the implied vs the true growth at the national level.

In [ ]:
amb.plot_national_quarterly_vs_implied()

Next let's look at what the regional growth (q on 4 q earlier) looks like for all regions.

In [ ]:
amb.plot_regional_annual_estimate()

If we want to just look at a specific region, we can. In the below, you can clearly see the nowcast period and where the model thinks regional growth will go based on national growth.

In [ ]:
amb.plot_single_region_annual_estimate(region_name="region_00")

We can zoom in on all the nowcasts, ie the latest period for which there are no annual publications:

In [ ]:
amb.plot_current_nowcast()

And, if we want tables of these, there's a built-in for that:

In [ ]:
amb.live_point_estimates().iloc[-6:, :]

You may think the point estimates are less useful than just whether an area is predicted to be in growth (2 periods successive predicted growth) or recession (2 periods negative growth) or indeterminate (neither.) There's also a built-in for this:

In [ ]:
amb.live_recession_indicator().iloc[-6:, :]